# Corpus documental y troceado

**Fase:** E10-E11

## Objetivo

Construir el corpus, trocearlo conservando fuente, sección y página, e indexarlo en PostgreSQL con pgvector.

## Relación con la pregunta de investigación

> ¿Puede un agente con RAG explicar el diagnóstico de un vehículo a partir de datos OBD-II, rastreando cada explicación hasta su evidencia de origen, y reconocer cuándo la evidencia disponible no es suficiente para sostener una explicación en lugar de generar una hipótesis sin fundamento?

Aporta la evidencia documental. Sin procedencia intacta no hay traza posible desde una afirmación hasta su origen.

## Hipótesis

_Por declarar antes de ejecutar la fase._

## Datos

_Dataset, versión y partición utilizados._

## Método

_Procedimiento, parámetros y semillas._

## Métricas

_Qué se mide y con qué instrumento._

## Criterio de éxito o refutación

_Fijado antes de mirar los resultados._


In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd

from mecanico_ia_exp.config import paths


## El corpus indexado

In [2]:
import sqlalchemy as sa
from mecanico_ia_exp.config.settings import get_settings

eng = sa.create_engine(get_settings().database.url)
with eng.connect() as cx:
    resumen = pd.read_sql(sa.text('''
        SELECT count(*) AS fragmentos,
               count(*) FILTER (WHERE cardinality(dtc_codes) > 0) AS con_dtc,
               count(DISTINCT source_file) AS ficheros,
               count(DISTINCT language) AS idiomas
        FROM rag.document_chunk'''), cx)
resumen

,fragmentos,con_dtc,ficheros,idiomas
0,6158,981,3055,1


### Procedencia completa de un fragmento

Nunca «según el manual» sin decir cuál.

In [3]:
from mecanico_ia_exp.rag.embeddings import get_provider
from mecanico_ia_exp.rag.store import PgVectorStore

store, prov = PgVectorStore(), get_provider('multilingual')
consulta = 'la sonda lambda responde con lentitud'
res = store.search(prov.embed_one(consulta), vehicle_id='mazda-3-bk-2007',
                   top_k=1, dtc_codes=('P0133',),
                   embedding_model=prov.identity.model)
for r in res:
    print('consulta :', consulta)
    print('cita     :', r.provenance())
    print('DTC      :', r.dtc_codes, '· motores:', r.engine_codes)

/Users/ldiaz/workspace/MasterAI/TFM/mecanico-ia-exp/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9135.62it/s]

consulta : la sonda lambda responde con lentitud
cita     : Mazda Motor Corporation · Mazda3 BK Workshop Manual (copia local, conversión de terceros) · sección «DTC P0133 [ZJ, Z6]» · página 1 · fragmento mazda3-bk-workshop-manual-local@9a90d2f6a951:001311 · https://www.mazdatechinfo.com/
DTC      : ('P0133',) · motores: ('Z6', 'ZJ')


### Aislamiento entre vehículos

In [4]:
otro = store.search(prov.embed_one(consulta), vehicle_id='otro-vehiculo',
                    top_k=5, embedding_model=prov.identity.model)
print(f'resultados para otro vehículo: {len(otro)}')
print('el vehículo es parámetro obligatorio: no hay búsqueda global')
store.dispose()

resultados para otro vehículo: 0
el vehículo es parámetro obligatorio: no hay búsqueda global


## Resultados

_Lo observado, sin interpretación._

## Interpretación

_Qué significa, y qué no significa._

## Limitaciones

_Qué queda fuera del alcance de lo mostrado._

## Conclusión experimental

_Respuesta al criterio fijado en la cabecera._

## Artefactos generados

_Ficheros producidos, con su ruta._
